# Genesis お手玉ロボット: PPO学習 (Google Colab)

2本のFranka Pandaアームが、お手玉に見立てた球2個を交換しあう方策をPPOで学習します。

**使い方**
1. メニューの「ランタイム」→「ランタイムのタイプを変更」で **GPU** を選択してください
2. 上から順にセルを実行してください
3. 学習中・学習後、Google Driveの出力フォルダ（`runs/<timestamp>/step_*/`）に
   `model_*.zip`（チェックポイント）・`reward_curve.png`（報酬推移）・`rollout_*.mp4`（現在の方策の動画）・
   `hyperparams.json`（使用したハイパーパラメータ）が定期的に保存されます。
4. パラメータ調整の相談をしたいときは、**そのフォルダをまるごとダウンロードしてClaudeにアップロード**してください。
   動画だけでなく報酬推移とハイパーパラメータも一緒に見られると、的確な提案がしやすくなります。

> 注記: このノートブックはローカル(CPU)では小規模に動作確認していますが、
> Colab GPU環境での実行そのものはまだ検証していません。エラーが出た場合は
> エラーメッセージごとClaudeに共有してください。

In [ ]:
!nvidia-smi

## 1. リポジトリの取得と依存関係のインストール

In [ ]:
import os

REPO_URL = "https://github.com/ky-ok/genesis-otedama-robot.git"
REPO_DIR = "/content/genesis-otedama-robot"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

In [ ]:
# レンダリング(EGL/OpenGL)に必要になることがある依存関係。
# 既に入っている場合は何もせず終わる。
!apt-get -qq update && apt-get -qq install -y libgl1 libegl1 libosmesa6 > /dev/null

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

## 2. Google Driveのマウント

チェックポイント・動画・報酬曲線をここに保存する。Colabのランタイムが切れても消えない。

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import datetime

RUN_NAME = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = f"/content/drive/MyDrive/genesis_otedama/runs/{RUN_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output dir:", OUTPUT_DIR)

## 3. Genesisの初期化とハイパーパラメータ

調整したいパラメータはこのセルにまとめている。ここを変えて再実行すれば別の設定で学習し直せる。

In [ ]:
import sys

sys.path.append(REPO_DIR)

import genesis as gs

gs.init(backend=gs.gpu)

In [ ]:
HYPERPARAMS = {
    # 環境
    "n_envs": 512,          # 並列シミュレーション数(GPUメモリに応じて調整)
    "dt": 0.02,
    # PPO
    "learning_rate": 3e-4,
    "n_steps": 64,          # 1回の更新までに各envが踏むステップ数
    "batch_size": 4096,
    "n_epochs": 10,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "clip_range": 0.2,
    "ent_coef": 0.0,
    # 学習の長さ
    "total_timesteps": 2_000_000,
    # チェックポイント/動画の保存間隔(タイムステップ数)
    "save_freq": 50_000,
    "rollout_steps": 300,
}
HYPERPARAMS

## 4. 学習

既存のチェックポイントから再開したい場合は `RESUME_FROM` にzipファイルのパスを入れる（新規学習ならNoneのまま）。

In [ ]:
from genesis_juggling.vec_env import JugglingVecEnv
from genesis_juggling.training_utils import CheckpointVideoCallback
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecMonitor

RESUME_FROM = None  # 例: f"{OUTPUT_DIR}/../20250101_120000/step_000500000/model_500000.zip"

vec_env = JugglingVecEnv(n_envs=HYPERPARAMS["n_envs"], dt=HYPERPARAMS["dt"])
vec_env = VecMonitor(vec_env)

if RESUME_FROM is not None:
    model = PPO.load(RESUME_FROM, env=vec_env)
else:
    model = PPO(
        "MlpPolicy",
        vec_env,
        learning_rate=HYPERPARAMS["learning_rate"],
        n_steps=HYPERPARAMS["n_steps"],
        batch_size=HYPERPARAMS["batch_size"],
        n_epochs=HYPERPARAMS["n_epochs"],
        gamma=HYPERPARAMS["gamma"],
        gae_lambda=HYPERPARAMS["gae_lambda"],
        clip_range=HYPERPARAMS["clip_range"],
        ent_coef=HYPERPARAMS["ent_coef"],
        verbose=1,
    )

callback = CheckpointVideoCallback(
    save_freq=HYPERPARAMS["save_freq"],
    output_dir=OUTPUT_DIR,
    hyperparams=HYPERPARAMS,
    rollout_steps=HYPERPARAMS["rollout_steps"],
)

model.learn(total_timesteps=HYPERPARAMS["total_timesteps"], callback=callback)

## 5. 最終チェックポイントの保存

学習が正常に終わった場合の最終モデルも明示的に保存しておく。

In [ ]:
model.save(f"{OUTPUT_DIR}/model_final.zip")
print("Saved final model to", f"{OUTPUT_DIR}/model_final.zip")